# Serve Qwen3-8B (vLLM + tunnel)

OpenAI-compatible API for agent trajectory generation. See [docs/setup.md](https://github.com/abdelmagid07/latent_failiure_prediction/blob/main/docs/setup.md).

**Runtime:** A100 GPU. Keep this tab open while running `devbugs_agent_colab.ipynb`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install cell
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'vllm==0.11.0',
    'transformers==4.55.4',
    'numpy<2.3',
], check=True)

# Verify the CUDA extension actually imports (fails fast if wheel mismatches runtime)
subprocess.run([sys.executable, '-c', 'import vllm._C; import vllm; print("vLLM", vllm.__version__)'], check=True)

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
MODEL = 'Qwen/Qwen3-8B'
SERVED_NAME = 'Qwen3-8B'
PORT = 8000
MAX_MODEL_LEN = 32768
API_KEY = 'EMPTY'

In [ ]:
# Launch cell 
import subprocess, sys

log = open('vllm.log', 'w')
cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL, '--served-model-name', SERVED_NAME,
    '--dtype', 'bfloat16', '--max-model-len', str(MAX_MODEL_LEN),
    '--port', str(PORT),
]
vllm_proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print('vLLM pid', vllm_proc.pid)

In [ ]:
import requests, time

url = f'http://localhost:{PORT}/v1/models'
for _ in range(120):
    if vllm_proc.poll() is not None:
        raise RuntimeError(open('vllm.log').read()[-3000:])
    try:
        if requests.get(url, timeout=5).ok:
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise TimeoutError('vLLM not ready')

In [ ]:
import subprocess, re, time

tunnel_log = open('cloudflared.log', 'w')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
public = None
for _ in range(60):
    time.sleep(2)
    if tunnel.poll() is not None:
        raise RuntimeError('cloudflared exited early:\n' + open('cloudflared.log').read()[-2000:])
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cloudflared.log').read())
    if m:
        public = m.group(0)
        break
if not public:
    tunnel.terminate()
    raise TimeoutError('No trycloudflare URL after 120s:\n' + open('cloudflared.log').read()[-2000:])
print(f'MODEL_API_BASE={public}/v1')

In [ ]:
import time
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    tunnel.terminate()
    vllm_proc.terminate()